# PySpark BigQuery Pipeline Demo

This notebook demonstrates how to use the PySpark BigQuery integration for data processing pipelines.

## Overview
- Configure Spark and BigQuery connections
- Read data from BigQuery tables
- Apply transformations using our pipeline utilities
- Write results back to BigQuery
- Perform data quality checks and analysis

## 1. Import Libraries and Setup

First, let's import all necessary libraries and set up our environment.

In [ ]:
import os
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

# Import our custom modules
from src.spark_session import get_spark_session
from src.bigquery_connector import BigQueryConnector
from src.pipelines.sample_pipeline import SamplePipeline
from src.transformations.common_transforms import (
    standardize_column_names,
    add_audit_columns,
    aggregate_metrics
)
from src.utils.data_validation import (
    validate_required_columns,
    check_data_quality,
    remove_duplicates
)
from config.logging import setup_logging, get_logger

# Standard data science libraries
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("✅ Libraries imported successfully!")


## 2. Configuration Setup

Configure logging and check our environment variables.

In [ ]:
# Setup logging for the notebook
setup_logging()
logger = get_logger("notebook-demo")

# Check environment variables
required_env_vars = [
    'GOOGLE_CLOUD_PROJECT_ID',
    'BIGQUERY_DATASET'
]

print("🔍 Checking environment configuration...")
for var in required_env_vars:
    value = os.getenv(var)
    if value:
        print(f"✅ {var}: {value}")
    else:
        print(f"⚠️ {var}: Not set (you may need to configure .env file)")

# Optional: Set environment variables if not configured
if not os.getenv('GOOGLE_CLOUD_PROJECT_ID'):
    print("\n📝 Setting demo environment variables...")
    os.environ['GOOGLE_CLOUD_PROJECT_ID'] = 'your-demo-project'
    os.environ['BIGQUERY_DATASET'] = 'demo_dataset'
    print("⚠️ Remember to update these with your actual project details!")

In [ ]:
import os
print(os.getenv("JAVA_HOME")) 

In [ ]:
from pyspark.sql import SparkSession
sc = SparkSession.builder.master("local[*]").getOrCreate()

## 3. Initialize Spark Session and BigQuery Connector

Create our Spark session with optimized configuration for BigQuery integration.

In [ ]:
# Initialize Spark session
print("🚀 Initializing Spark session...")
spark = get_spark_session("notebook-demo")

# Initialize BigQuery connector
bq_connector = BigQueryConnector(spark)

# Display Spark configuration
print(f"✅ Spark session created: {spark.conf.get('spark.app.name')}")
print(f"🔧 Spark master: {spark.conf.get('spark.master')}")
print(f"💾 Warehouse dir: {spark.conf.get('spark.sql.warehouse.dir')}")

# Show Spark UI URL
print(f"\n🌐 Spark UI available at: http://localhost:4040")

## 4. Create Sample Data

Since we might not have actual BigQuery tables yet, let's create some sample data to work with.

In [ ]:
# Create sample data for demonstration
from datetime import datetime, timedelta
import random

print("📊 Creating sample data...")

# Generate sample data
sample_data = []
for i in range(1000):
    sample_data.append({
        'id': i + 1,
        'customer_name': f'Customer {i+1}',
        'email': f'customer{i+1}@example.com',
        'age': random.randint(18, 80),
        'city': random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']),
        'purchase_amount': round(random.uniform(10.0, 1000.0), 2),
        'purchase_date': (datetime.now() - timedelta(days=random.randint(0, 365))).date(),
        'category': random.choice(['Electronics', 'Clothing', 'Books', 'Home', 'Sports'])
    })

# Create Spark DataFrame
schema = StructType([
    StructField('id', IntegerType(), False),
    StructField('customer_name', StringType(), True),
    StructField('email', StringType(), True),
    StructField('age', IntegerType(), True),
    StructField('city', StringType(), True),
    StructField('purchase_amount', DoubleType(), True),
    StructField('purchase_date', DateType(), True),
    StructField('category', StringType(), True)
])

df = sc.createDataFrame(sample_data, schema)

print(f"✅ Created sample DataFrame with {df.count()} rows")
print(f"📋 Columns: {df.columns}")

# Display first few rows
df.show(5, truncate=False)

## 5. Data Quality Assessment

Let's check the quality of our data using our validation utilities.

In [ ]:
print("🔍 Performing data quality checks...")

# Validate required columns
required_columns = ['id', 'customer_name', 'email']
try:
    validate_required_columns(df, required_columns)
    print("✅ All required columns present")
except ValueError as e:
    print(f"❌ Validation failed: {e}")

# Check data quality
quality_report = check_data_quality(df)

print(f"\n📊 Data Quality Report:")
print(f"Total rows: {quality_report['total_rows']}")
print("\nColumn Quality:")
for col, stats in quality_report['columns'].items():
    print(f"  {col:15} | Nulls: {stats['null_count']:4d} ({stats['null_percentage']:5.1f}%) | "
          f"Distinct: {stats['distinct_count']:4d} | Completeness: {stats['completeness']:5.1f}%")

## 6. Data Transformations

Apply common transformations to our dataset.

In [ ]:
print("🔄 Applying data transformations...")

# Start with our base DataFrame
transformed_df = df

# 1. Standardize column names (convert to snake_case)
print("📝 Standardizing column names...")
# Note: Our columns are already in snake_case, but this shows the process
transformed_df = standardize_column_names(transformed_df, "snake_case")

# 2. Add audit columns for data lineage
print("📅 Adding audit columns...")
transformed_df = add_audit_columns(transformed_df)

# 3. Create derived columns
print("🎯 Creating derived columns...")
transformed_df = transformed_df.withColumn(
    "age_group",
    F.when(F.col("age") < 25, "Young")
    .when(F.col("age") < 50, "Middle-aged")
    .otherwise("Senior")
)

transformed_df = transformed_df.withColumn(
    "purchase_tier",
    F.when(F.col("purchase_amount") < 100, "Low")
    .when(F.col("purchase_amount") < 500, "Medium")
    .otherwise("High")
)

# 4. Add date dimensions
print("📊 Adding date dimensions...")
transformed_df = transformed_df.withColumn(
    "purchase_year", F.year(F.col("purchase_date"))
).withColumn(
    "purchase_month", F.month(F.col("purchase_date"))
).withColumn(
    "purchase_quarter", F.quarter(F.col("purchase_date"))
)

print(f"✅ Transformation complete. New columns: {len(transformed_df.columns)}")
print(f"📋 All columns: {transformed_df.columns}")

# Show sample of transformed data
transformed_df.select(
    "id", "customer_name", "age", "age_group", 
    "purchase_amount", "purchase_tier", "load_timestamp"
).show(5)

## 7. Data Analysis and Aggregation

Perform some basic analysis using our aggregation utilities.

In [ ]:
print("📈 Performing data analysis...")

# 1. Aggregate by age group
print("\n👥 Customer analysis by age group:")
age_group_metrics = aggregate_metrics(
    transformed_df,
    group_by_columns=["age_group"],
    metrics={
        "id": "count",
        "purchase_amount": "sum",
        "purchase_amount": "avg"
    }
)
age_group_metrics.orderBy("id_count", ascending=False).show()

# 2. Aggregate by city and category
print("\n🏙️ Sales analysis by city and category:")
city_category_metrics = (
    transformed_df
    .groupBy("city", "category")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("purchase_amount").alias("total_sales"),
        F.avg("purchase_amount").alias("avg_purchase"),
        F.max("purchase_amount").alias("max_purchase")
    )
    .orderBy("total_sales", ascending=False)
)

city_category_metrics.show(10, truncate=False)

# 3. Monthly sales trend
print("\n📅 Monthly sales trend:")
monthly_sales = (
    transformed_df
    .groupBy("purchase_year", "purchase_month")
    .agg(
        F.count("*").alias("transactions"),
        F.sum("purchase_amount").alias("total_sales"),
        F.avg("purchase_amount").alias("avg_transaction")
    )
    .orderBy("purchase_year", "purchase_month")
)

monthly_sales.show()

## 8. Using the Sample Pipeline Class

Now let's demonstrate how to use our pre-built pipeline class.

In [ ]:
print("🔧 Demonstrating Sample Pipeline class...")

# Initialize the pipeline
pipeline = SamplePipeline("notebook-pipeline-demo")

# Since we don't have actual BigQuery tables, we'll demonstrate the transformation logic
print("\n🔄 Testing pipeline transformations...")

# Apply the pipeline transformation logic to our sample data
pipeline_result = pipeline.transform_data(df)

print(f"✅ Pipeline transformation complete!")
print(f"📊 Input rows: {df.count()}")
print(f"📊 Output rows: {pipeline_result.count()}")
print(f"📋 Output columns: {len(pipeline_result.columns)}")

# Show the pipeline result
pipeline_result.select(
    "id", "customer_name", "purchase_amount", 
    "load_timestamp", "source_system"
).show(5)

## 9. Reading from BigQuery (Demo)

Here's how you would read data from actual BigQuery tables once configured.

In [ ]:
# This cell demonstrates BigQuery reading - uncomment when you have actual tables

print("📚 BigQuery Reading Examples (commented out for demo):")

# Example 1: Read entire table
print("\n1. Reading entire table:")
print("# df_from_bq = bq_connector.read_table('your_dataset.your_table')")

# Example 2: Read with column selection
print("\n2. Reading specific columns:")
print("# df_subset = bq_connector.read_table(")
print("#     table_id='your_dataset.your_table',")
print("#     columns=['id', 'name', 'email', 'created_date']")
print("# )")

# Example 3: Read with filter
print("\n3. Reading with filter condition:")
print("# df_filtered = bq_connector.read_table(")
print("#     table_id='your_dataset.your_table',")
print("#     filter_condition=\"created_date >= '2024-01-01'\"")
print("# )")

# Example 4: Execute custom query
print("\n4. Execute custom SQL query:")
print("# custom_query = \"\"\"")
print("# SELECT customer_id, SUM(amount) as total_amount")
print("# FROM `project.dataset.transactions`")
print("# WHERE transaction_date >= '2024-01-01'")
print("# GROUP BY customer_id")
print("# HAVING total_amount > 100")
print("# \"\"\"")
print("# df_query_result = bq_connector.execute_query(custom_query)")

print("\n💡 To use these examples:")
print("   1. Set up your BigQuery credentials")
print("   2. Update the table names with your actual BigQuery tables")
print("   3. Uncomment the code above")

## 10. Writing to BigQuery (Demo)

Here's how you would write results back to BigQuery.

In [ ]:
print("💾 BigQuery Writing Examples (commented out for demo):")

# Example 1: Write DataFrame to BigQuery
print("\n1. Write DataFrame to BigQuery:")
print("# bq_connector.write_table(")
print("#     df=transformed_df,")
print("#     table_id='your_dataset.processed_customers',")
print("#     mode='overwrite'")
print("# )")

# Example 2: Append to existing table
print("\n2. Append to existing table:")
print("# bq_connector.write_table(")
print("#     df=new_data,")
print("#     table_id='your_dataset.daily_sales',")
print("#     mode='append'")
print("# )")

# Example 3: Write with partitioning
print("\n3. Write with date partitioning:")
print("# bq_connector.write_table(")
print("#     df=transformed_df,")
print("#     table_id='your_dataset.partitioned_data',")
print("#     mode='overwrite',")
print("#     partition_by='load_timestamp'")
print("# )")

# Example 4: Write with clustering
print("\n4. Write with clustering:")
print("# bq_connector.write_table(")
print("#     df=transformed_df,")
print("#     table_id='your_dataset.clustered_data',")
print("#     mode='overwrite',")
print("#     cluster_by=['city', 'category']")
print("# )")

print("\n📝 For demonstration, let's save our data locally:")

# Save to local parquet file for demonstration
output_path = "../data/processed_sample_data"
print(f"💾 Saving processed data to: {output_path}")

# Write to parquet format
transformed_df.write.mode("overwrite").parquet(output_path)

print("✅ Data saved successfully!")
print(f"📊 Records saved: {transformed_df.count()}")

## 11. Performance Monitoring

Let's look at some basic performance metrics and Spark execution details.

In [ ]:
print("📊 Performance Monitoring:")

# Get Spark context for metrics
sc = spark.sparkContext

print(f"\n🔧 Spark Configuration:")
print(f"   App Name: {spark.conf.get('spark.app.name')}")
print(f"   Master: {spark.conf.get('spark.master')}")
print(f"   Driver Memory: {spark.conf.get('spark.driver.memory')}")
print(f"   Executor Memory: {spark.conf.get('spark.executor.memory')}")

print(f"\n📈 Execution Metrics:")
print(f"   Default Parallelism: {sc.defaultParallelism}")
print(f"   Active Jobs: {len(sc.statusTracker().getActiveJobIds())}")
print(f"   Active Stages: {len(sc.statusTracker().getActiveStageIds())}")

# DataFrame statistics
print(f"\n📊 DataFrame Cache Status:")
cached_tables = spark.catalog.listTables()
if cached_tables:
    for table in cached_tables:
        print(f"   Table: {table.name}, Cached: {table.isTemporary}")
else:
    print("   No cached tables")

print(f"\n🌐 Access Spark UI at: http://localhost:4040")
print("   - Jobs tab: See job execution details")
print("   - Stages tab: See stage-level metrics")
print("   - Storage tab: See cached RDD/DataFrame info")
print("   - Environment tab: See all Spark configurations")

## 12. Cleanup

Clean up resources and provide next steps.

In [ ]:
print("🧹 Cleanup and Summary:")

# Unpersist any cached DataFrames (if we had cached them)
# transformed_df.unpersist()

print("\n✅ Notebook execution complete!")
print("\n📋 What we accomplished:")
print("   ✓ Set up Spark session with BigQuery integration")
print("   ✓ Created and processed sample data")
print("   ✓ Applied data quality checks and transformations")
print("   ✓ Performed data analysis and aggregations")
print("   ✓ Demonstrated pipeline usage patterns")
print("   ✓ Showed BigQuery read/write examples")

print("\n🚀 Next Steps:")
print("   1. Configure your actual BigQuery credentials")
print("   2. Replace sample data with your actual BigQuery tables")
print("   3. Customize transformations for your specific use case")
print("   4. Set up automated pipeline execution")
print("   5. Monitor performance using Spark UI and BigQuery console")

print("\n💡 Tips:")
print("   - Use .cache() for DataFrames used multiple times")
print("   - Partition large tables by date for better performance")
print("   - Monitor BigQuery slot usage and costs")
print("   - Use broadcast joins for small lookup tables")

print("\n📚 Resources:")
print("   - Project README: ../README.md")
print("   - Configuration: ../config/settings.py")
print("   - Pipeline examples: ../src/pipelines/")
print("   - Spark UI: http://localhost:4040")

# Note: Spark session will remain active for continued work
print("\n⚡ Spark session remains active for continued development.")
print("   Call spark.stop() when completely finished.")